# 03 CleanData — PalletLogs

Notebook นี้ใช้สำหรับ clean ข้อมูลต่อจากไฟล์ transformed

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

## Load Transformed Data

เริ่มจากไฟล์ `palletlogs_transformed.csv` ที่ได้จากขั้น transform

In [2]:
source_path = '../../data/interim/palletlogs/palletlogs_transformed.csv'
df = pd.read_csv(source_path, encoding='utf-8-sig', parse_dates=['StampDateTime'])

print(f'source: {source_path}')
print(f'shape before clean: {df.shape}')
df.head()

source: ../../data/interim/palletlogs/palletlogs_transformed.csv
shape before clean: (399233, 28)


,PalletID,Action,PalletStatus,PackSize,Amount,LotNo,MaterialCode,NameThai,ProductType,Model,...,FromRowNo,FromColNo,FromFloorNo,ToLocationType,ToLocationCode,ToLocationName,ToColNo,ToFloorNo,UserCreate,UserCreateCode
0,SB1LR260511450,POSTTOCHECKERLOCATION,FOLKPOSTEND,0,0,NaN,NaN,NaN,NaN,NaN,...,NaN,0,0,POSTLOCATION,COM22090001,SB1-1ลานโอน,0,0,&#3616;&#3588;&#3623;&#3633;&#3605; &#3648;&#3...,2503000002
1,SB1LR260511451,POSTTOCHECKERLOCATION,FOLKPOSTEND,0,0,NaN,NaN,NaN,NaN,NaN,...,NaN,0,0,POSTLOCATION,COM23110002,ลานจ่าย3 ช่องจ่าย 2 (ครอบ),0,0,&#3616;&#3588;&#3623;&#3633;&#3605; &#3648;&#3...,2503000002
2,SB1-1LS3690515P025,POSTTOCHECKERLOCATION,FOLKPOSTEND,320,320,3690515P,ZCB10STDA002000A14,ก/บ คอนกรีตSCGเอลาบานา สีน้ำตาลโอ๊คแดง,CPAC-Tile,CPAC,...,48.0,23,1,POSTLOCATION,PL20100001,ลานจ่าย 1 ช่องจ่าย 1 (CPAC),0,0,ชัยณรงค์ เพ็ชรนิล,2102000002
3,SB1LR260511214,POSTTOCHECKERLOCATION,FOLKPOSTEND,0,0,NaN,NaN,NaN,NaN,NaN,...,NaN,0,0,POSTLOCATION,COM23110003,อุปกรณ์,0,0,เสน่ห์ แสงตระกอน,2202000004
4,NaN,POSTTOCHECKERLOCATION,NaN,0,0,NaN,NaN,NaN,NaN,NaN,...,NaN,0,0,POSTLOCATION,PL20100001,ลานจ่าย 1 ช่องจ่าย 1 (CPAC),0,0,ชัยณรงค์ เพ็ชรนิล,2102000002


## Step 1: ลบแถวที่ PalletID เป็น null

`PalletID` เป็น identifier หลักของ pallet — แถวที่ไม่มี PalletID ไม่สามารถระบุตัวตนได้

In [3]:
df_clean = df.copy()
rows_before = len(df_clean)

df_clean = df_clean.loc[df_clean['PalletID'].notna()].copy()
rows_after = len(df_clean)

print(f'rows before: {rows_before:,}')
print(f'rows after:  {rows_after:,}')
print(f'rows removed (null PalletID): {rows_before - rows_after:,}')

rows before: 399,233
rows after:  351,947
rows removed (null PalletID): 47,286


## Step 2: กรอง StampDateTime ที่ parse ไม่ได้

ลบแถวที่ `StampDateTime` เป็น NaT (parse ล้มเหลว)

In [4]:
rows_before = len(df_clean)

df_clean = df_clean.loc[df_clean['StampDateTime'].notna()].copy()
rows_after = len(df_clean)

print(f'rows removed (null StampDateTime): {rows_before - rows_after:,}')
print(f'rows remaining: {rows_after:,}')

rows removed (null StampDateTime): 0
rows remaining: 351,947


## Step 3: กรองเฉพาะปี 2025 เป็นต้นไป

เก็บเฉพาะข้อมูลปี 2025+ เพื่อให้สอดคล้องกับช่วงที่ระบบทำงานปกติ

In [5]:
rows_before = len(df_clean)

year_mask = df_clean['StampDateTime'].dt.year >= 2025
df_clean = df_clean.loc[year_mask].copy()

rows_after = len(df_clean)
print(f'rows removed (before 2025): {rows_before - rows_after:,}')
print(f'rows remaining: {rows_after:,}')

print('\nYear distribution after filter:')
print(df_clean['StampDateTime'].dt.year.value_counts().sort_index())

rows removed (before 2025): 0
rows remaining: 351,947

Year distribution after filter:
StampDateTime
2025    249933
2026    102014
Name: count, dtype: int64


## Step 4: กรอง FromLocationType

เก็บเฉพาะ `FromLocationType` ที่เป็น NORMAL และ BUFFER (การเคลื่อน pallet จริงจากคลัง)
- `PALLETSOURCE` คือแถวที่ pallet เพิ่งถูก pack มาใหม่ ยังไม่ได้อยู่ใน location จริง

In [6]:
print('FromLocationType distribution before filter:')
print(df_clean['FromLocationType'].value_counts(dropna=False))

rows_before = len(df_clean)
keep_from_types = ['NORMAL', 'BUFFER', 'FRACTION']
from_type_mask = df_clean['FromLocationType'].isin(keep_from_types)
df_clean = df_clean.loc[from_type_mask].copy()
rows_after = len(df_clean)

print(f'\nrows removed (FromLocationType not in {keep_from_types}): {rows_before - rows_after:,}')
print(f'rows remaining: {rows_after:,}')

FromLocationType distribution before filter:
FromLocationType
NORMAL          278397
PALLETSOURCE     42846
BUFFER           30686
FRACTION            17
NaN                  1
Name: count, dtype: int64

rows removed (FromLocationType not in ['NORMAL', 'BUFFER', 'FRACTION']): 42,847
rows remaining: 309,100


## Step 5: กรอง Amount ที่เป็น 0 หรือ negative

แถวที่ `Amount <= 0` ไม่มีความหมายด้านปริมาณ

In [7]:
rows_before = len(df_clean)

df_clean = df_clean.loc[df_clean['Amount'] > 0].copy()
rows_after = len(df_clean)

print(f'rows removed (Amount <= 0): {rows_before - rows_after:,}')
print(f'rows remaining: {rows_after:,}')

rows removed (Amount <= 0): 0
rows remaining: 309,100


## Step 6: ตรวจ Duplicate

ตรวจหาแถวซ้ำด้วย key = `PalletID + StampDateTime`

In [8]:
dup_key = ['PalletID', 'StampDateTime']
dup_count = df_clean.duplicated(subset=dup_key).sum()
print(f'duplicate rows (PalletID + StampDateTime): {dup_count:,}')

if dup_count > 0:
    rows_before = len(df_clean)
    df_clean = df_clean.drop_duplicates(subset=dup_key, keep='first').copy()
    print(f'rows removed: {rows_before - len(df_clean):,}')
    print(f'rows remaining: {len(df_clean):,}')
else:
    print('no duplicates found')

duplicate rows (PalletID + StampDateTime): 36
rows removed: 36
rows remaining: 309,064


## Removal Summary

สรุปจำนวนแถวที่ถูกตัดออกแยกตามเหตุผลในแต่ละ step

In [9]:
raw_rows = len(df)
final_rows = len(df_clean)

print(f'raw rows      : {raw_rows:,}')
print(f'final rows    : {final_rows:,}')
print(f'total removed : {raw_rows - final_rows:,} ({(raw_rows - final_rows)/raw_rows*100:.1f}%)')

print('\n--- Clean Data Summary ---')
print(f'shape: {df_clean.shape}')
print(f'date range: {df_clean["StampDateTime"].min()} → {df_clean["StampDateTime"].max()}')
print(f'\nFromLocationType:')
print(df_clean['FromLocationType'].value_counts())
print(f'\nProductType:')
print(df_clean['ProductType'].value_counts(dropna=False))

raw rows      : 399,233
final rows    : 309,064
total removed : 90,169 (22.6%)

--- Clean Data Summary ---
shape: (309064, 28)
date range: 2025-01-02 07:30:08 → 2026-05-18 10:51:20

FromLocationType:
FromLocationType
NORMAL      278361
BUFFER       30686
FRACTION        17
Name: count, dtype: int64

ProductType:
ProductType
CPAC-Tile           152475
PRESTIGE-Tile       148301
CPAC-Fitting          5453
PRESTIGE-Fitting      2743
DURA-Fitting            92
Name: count, dtype: int64


## Preview Cleaned Data

In [10]:
df_clean.head(10)

,PalletID,Action,PalletStatus,PackSize,Amount,LotNo,MaterialCode,NameThai,ProductType,Model,...,FromRowNo,FromColNo,FromFloorNo,ToLocationType,ToLocationCode,ToLocationName,ToColNo,ToFloorNo,UserCreate,UserCreateCode
2,SB1-1LS3690515P025,POSTTOCHECKERLOCATION,FOLKPOSTEND,320,320,3690515P,ZCB10STDA002000A14,ก/บ คอนกรีตSCGเอลาบานา สีน้ำตาลโอ๊คแดง,CPAC-Tile,CPAC,...,48.0,23,1,POSTLOCATION,PL20100001,ลานจ่าย 1 ช่องจ่าย 1 (CPAC),0,0,ชัยณรงค์ เพ็ชรนิล,2102000002
5,SB1-1LS3690515P026,POSTTOCHECKERLOCATION,FOLKPOSTEND,320,320,3690515P,ZCB10STDA002000A14,ก/บ คอนกรีตSCGเอลาบานา สีน้ำตาลโอ๊คแดง,CPAC-Tile,CPAC,...,48.0,23,2,POSTLOCATION,PL20100001,ลานจ่าย 1 ช่องจ่าย 1 (CPAC),0,0,ชัยณรงค์ เพ็ชรนิล,2102000002
6,SB1-1LS3690515P024,POSTTOCHECKERLOCATION,FOLKPOSTEND,320,320,3690515P,ZCB10STDA002000A14,ก/บ คอนกรีตSCGเอลาบานา สีน้ำตาลโอ๊คแดง,CPAC-Tile,CPAC,...,48.0,22,2,POSTLOCATION,PL20100001,ลานจ่าย 1 ช่องจ่าย 1 (CPAC),0,0,&#3626;&#3640;&#3614;&#3633;&#3602;&#3609;&#36...,2506000001
8,SB1-1LS3690515P023,POSTTOCHECKERLOCATION,FOLKPOSTEND,320,320,3690515P,ZCB10STDA002000A14,ก/บ คอนกรีตSCGเอลาบานา สีน้ำตาลโอ๊คแดง,CPAC-Tile,CPAC,...,48.0,22,1,POSTLOCATION,PL20100001,ลานจ่าย 1 ช่องจ่าย 1 (CPAC),0,0,พัสกร จันหอม,2102000001
11,SB1-1LS3690515P021,POSTTOCHECKERLOCATION,FOLKPOSTEND,320,320,3690515P,ZCB10STDA002000A14,ก/บ คอนกรีตSCGเอลาบานา สีน้ำตาลโอ๊คแดง,CPAC-Tile,CPAC,...,48.0,21,1,POSTLOCATION,PL20100001,ลานจ่าย 1 ช่องจ่าย 1 (CPAC),0,0,&#3626;&#3640;&#3614;&#3633;&#3602;&#3609;&#36...,2506000001
13,SB1-1LS3690515P022,POSTTOCHECKERLOCATION,FOLKPOSTEND,320,320,3690515P,ZCB10STDA002000A14,ก/บ คอนกรีตSCGเอลาบานา สีน้ำตาลโอ๊คแดง,CPAC-Tile,CPAC,...,48.0,21,2,POSTLOCATION,PL20100001,ลานจ่าย 1 ช่องจ่าย 1 (CPAC),0,0,&#3626;&#3640;&#3614;&#3633;&#3602;&#3609;&#36...,2506000001
15,SB1-1LS3690515P020,POSTTOCHECKERLOCATION,FOLKPOSTEND,320,320,3690515P,ZCB10STDA002000A14,ก/บ คอนกรีตSCGเอลาบานา สีน้ำตาลโอ๊คแดง,CPAC-Tile,CPAC,...,48.0,20,2,POSTLOCATION,PL20100001,ลานจ่าย 1 ช่องจ่าย 1 (CPAC),0,0,&#3626;&#3640;&#3614;&#3633;&#3602;&#3609;&#36...,2506000001
16,SB1-1LS3690515P019,POSTTOCHECKERLOCATION,FOLKPOSTEND,320,320,3690515P,ZCB10STDA002000A14,ก/บ คอนกรีตSCGเอลาบานา สีน้ำตาลโอ๊คแดง,CPAC-Tile,CPAC,...,48.0,20,1,POSTLOCATION,PL20100001,ลานจ่าย 1 ช่องจ่าย 1 (CPAC),0,0,ชัยณรงค์ เพ็ชรนิล,2102000002
17,SB1-1LS3690515P017,POSTTOCHECKERLOCATION,FOLKPOSTEND,320,320,3690515P,ZCB10STDA002000A14,ก/บ คอนกรีตSCGเอลาบานา สีน้ำตาลโอ๊คแดง,CPAC-Tile,CPAC,...,48.0,13,1,POSTLOCATION,PL20100001,ลานจ่าย 1 ช่องจ่าย 1 (CPAC),0,0,&#3626;&#3640;&#3614;&#3633;&#3602;&#3609;&#36...,2506000001
19,SB1-1LS3690515P018,POSTTOCHECKERLOCATION,FOLKPOSTEND,320,320,3690515P,ZCB10STDA002000A14,ก/บ คอนกรีตSCGเอลาบานา สีน้ำตาลโอ๊คแดง,CPAC-Tile,CPAC,...,48.0,17,1,POSTLOCATION,PL20100001,ลานจ่าย 1 ช่องจ่าย 1 (CPAC),0,0,ชัยณรงค์ เพ็ชรนิล,2102000002


## Save Clean Data

บันทึกผล clean ไว้ใน `data/interim/palletlogs` เพื่อใช้วิเคราะห์ต่อ

In [11]:
output_path = '../../data/interim/palletlogs/palletlogs_clean.csv'
df_clean.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'saved: {output_path}')
print(f'shape: {df_clean.shape}')
output_path

saved: ../../data/interim/palletlogs/palletlogs_clean.csv
shape: (309064, 28)


'../../data/interim/palletlogs/palletlogs_clean.csv'